# Oral squamous cell carcinoma · GSE208253

This spatial transcriptomics notebook uses the dataset-specific marker panel in the cells below. Prepare the inputs listed in [the data guide](../../data/README.md). Start with a fresh kernel and run the annotation cells in order. The last section contains optional comparison methods. The notebook writes its files under `results/runs/spatial/oral_gse208253/`.


In [ ]:
from pathlib import Path
import os

# Both a kernel started alongside the notebook and one started at the repository root work.
start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p / "data" / "datasets.tsv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from a checkout containing data/datasets.tsv.")
RAW = ROOT / "data" / "raw"
PREPARED = ROOT / "data" / "prepared"
RUN_DIR = ROOT / "results" / "runs" / "spatial" / "oral_gse208253"
RUN_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(RUN_DIR)
RUN_COMPARISONS = False     # Enable after installing the optional comparison tools.


In [ ]:
# ============================================================
# GSE208253: 12 oral squamous cell carcinoma Visium samples
#
# Expected files:
# GSM6339631_s1_filtered_feature_bc_matrix.h5
# GSM6339631_s1_tissue_positions_list.csv
# ...
# GSM6339642_s12_filtered_feature_bc_matrix.h5
# GSM6339642_s12_tissue_positions_list.csv
# ============================================================

from pathlib import Path
import gzip
import shutil
import json
import re

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad


# ------------------------------------------------------------
# 1. Folder containing all extracted GSE208253 files
# ------------------------------------------------------------

data_dir = RAW / "GSE208253_RAW"

# Temporary folder used only when an H5 file remains gzip-compressed
cache_dir = data_dir / "_decompressed_h5"
cache_dir.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. Helpers
# ------------------------------------------------------------

def natural_sort_key(path):
    """Sort s1, s2, ..., s10, s11, s12 correctly."""
    text = str(path)

    return [
        int(token) if token.isdigit() else token.lower()
        for token in re.split(r"(\d+)", text)
    ]


def decompress_h5_if_needed(h5_path, cache_dir):
    """
    sc.read_10x_h5 cannot directly read an .h5.gz file.
    Decompress it once and reuse the decompressed copy.
    """

    h5_path = Path(h5_path)

    if h5_path.suffix != ".gz":
        return h5_path

    output_name = h5_path.name[:-3]  # remove .gz
    output_path = cache_dir / output_name

    if not output_path.exists():

        print(f"Decompressing: {h5_path.name}")

        with gzip.open(h5_path, "rb") as source:
            with open(output_path, "wb") as destination:
                shutil.copyfileobj(source, destination)

    return output_path


def find_matching_file(folder, patterns):
    """Return the first existing file matching one of the patterns."""

    matches = []

    for pattern in patterns:
        matches.extend(folder.glob(pattern))

    matches = sorted(
        set(matches),
        key=natural_sort_key
    )

    return matches[0] if matches else None


def extract_sample_information(filename):
    """
    Example:
    GSM6339631_s1_filtered_feature_bc_matrix.h5
    """

    match = re.search(
        r"(GSM\d+)_(s\d+)_",
        filename
    )

    if match is None:
        raise ValueError(
            f"Could not extract GSM/sample name from: {filename}"
        )

    gsm = match.group(1)
    sample = match.group(2)

    return gsm, sample


# ------------------------------------------------------------
# 3. Discover the 12 H5 files
# ------------------------------------------------------------

h5_files = list(
    data_dir.glob("*_filtered_feature_bc_matrix.h5")
)

h5_files += list(
    data_dir.glob("*_filtered_feature_bc_matrix.h5.gz")
)

h5_files = sorted(
    h5_files,
    key=natural_sort_key
)

print("Number of H5 files found:", len(h5_files))

for path in h5_files:
    print(path.name)

if len(h5_files) != 12:
    print(
        "\nWarning: GSE208253 should contain 12 spatial samples, "
        f"but {len(h5_files)} H5 files were found."
    )

In [ ]:
def read_gse208253_sample(
    h5_source,
    data_dir,
    cache_dir,
    tissue_only=True
):
    """
    Read one GSE208253 Visium sample from:
      - filtered_feature_bc_matrix.h5
      - tissue_positions_list.csv

    Optionally attaches:
      - tissue_hires_image.png
      - scalefactors_json.json
    """

    h5_source = Path(h5_source)

    gsm, sample_name = extract_sample_information(
        h5_source.name
    )

    sample_id = f"{gsm}_{sample_name}"

    # --------------------------------------------------------
    # A. Read expression matrix
    # --------------------------------------------------------

    h5_path = decompress_h5_if_needed(
        h5_source,
        cache_dir
    )

    adata = sc.read_10x_h5(
        h5_path,
        genome=None,
        gex_only=True
    )

    adata.var_names_make_unique()

    # Preserve original Visium barcode
    adata.obs["barcode"] = adata.obs_names.astype(str)

    # --------------------------------------------------------
    # B. Find tissue-position file
    # --------------------------------------------------------

    position_path = find_matching_file(
        data_dir,
        [
            f"{gsm}_{sample_name}_tissue_positions_list.csv",
            f"{gsm}_{sample_name}_tissue_positions_list.csv.gz",
            f"{gsm}_{sample_name}_tissue_positions.csv",
            f"{gsm}_{sample_name}_tissue_positions.csv.gz",
        ]
    )

    if position_path is None:
        raise FileNotFoundError(
            f"No tissue-position file found for {sample_id}"
        )

    # GSE208253 files use the older six-column, header-free format:
    #
    # barcode
    # in_tissue
    # array_row
    # array_col
    # pxl_row_in_fullres
    # pxl_col_in_fullres

    positions = pd.read_csv(
        position_path,
        header=None,
        compression="infer"
    )

    if positions.shape[1] != 6:
        raise ValueError(
            f"{position_path.name} has {positions.shape[1]} columns; "
            "six columns were expected."
        )

    positions.columns = [
        "barcode",
        "in_tissue",
        "array_row",
        "array_col",
        "pxl_row_in_fullres",
        "pxl_col_in_fullres",
    ]

    positions["barcode"] = positions["barcode"].astype(str)

    positions = positions.set_index(
        "barcode"
    )

    # The H5 matrix contains filtered barcodes.
    # Reindex the full position table to those H5 barcodes.
    positions = positions.reindex(
        adata.obs["barcode"]
    )

    missing_positions = positions[
        "pxl_row_in_fullres"
    ].isna().sum()

    if missing_positions > 0:
        raise ValueError(
            f"{sample_id}: {missing_positions} expression barcodes "
            "did not match the tissue-position file."
        )

    # --------------------------------------------------------
    # C. Store spatial metadata
    # --------------------------------------------------------

    coordinate_columns = [
        "in_tissue",
        "array_row",
        "array_col",
        "pxl_row_in_fullres",
        "pxl_col_in_fullres",
    ]

    for column in coordinate_columns:
        adata.obs[column] = positions[column].to_numpy()

    # Scanpy/stLearn convention:
    # first column = x = pixel column
    # second column = y = pixel row
    adata.obsm["spatial"] = positions[
        [
            "pxl_col_in_fullres",
            "pxl_row_in_fullres",
        ]
    ].to_numpy(dtype=np.float64)

    # Grid coordinates can also be useful
    adata.obsm["array_coordinates"] = positions[
        [
            "array_col",
            "array_row",
        ]
    ].to_numpy(dtype=np.float64)

    # --------------------------------------------------------
    # D. Add sample metadata
    # --------------------------------------------------------

    adata.obs["sample"] = sample_name
    adata.obs["gsm"] = gsm
    adata.obs["sample_id"] = sample_id
    adata.obs["condition"] = "OSCC"
    adata.obs["cancer_type"] = "Oral_squamous_cell_carcinoma"
    adata.obs["dataset"] = "GSE208253"

    # Unique observation names across all 12 samples
    adata.obs_names = pd.Index(
        [
            f"{sample_name}_{barcode}"
            for barcode in adata.obs["barcode"].astype(str)
        ]
    )

    # --------------------------------------------------------
    # E. Optionally retain only tissue-covered spots
    # --------------------------------------------------------

    if tissue_only:
        adata = adata[
            adata.obs["in_tissue"].astype(int) == 1
        ].copy()

    # --------------------------------------------------------
    # F. Optional image and scale factors
    # --------------------------------------------------------

    image_path = find_matching_file(
        data_dir,
        [
            f"{gsm}_{sample_name}_tissue_hires_image.png",
            f"{gsm}_{sample_name}_tissue_hires_image.png.gz",
        ]
    )

    scalefactor_path = find_matching_file(
        data_dir,
        [
            f"{gsm}_{sample_name}_scalefactors_json.json",
            f"{gsm}_{sample_name}_scalefactors_json.json.gz",
            f"{gsm}_{sample_name}_scalefactors_json.json",
        ]
    )

    adata.uns.setdefault("spatial", {})
    adata.uns["spatial"][sample_id] = {
        "metadata": {
            "dataset": "GSE208253",
            "gsm": gsm,
            "sample": sample_name,
        }
    }

    # Read image when available
    if image_path is not None:

        from PIL import Image

        if image_path.suffix == ".gz":

            with gzip.open(image_path, "rb") as handle:
                image = np.asarray(
                    Image.open(handle).convert("RGB")
                )

        else:
            image = np.asarray(
                Image.open(image_path).convert("RGB")
            )

        adata.uns["spatial"][sample_id]["images"] = {
            "hires": image
        }

    # Read scale factors when available
    if scalefactor_path is not None:

        if scalefactor_path.suffix == ".gz":
            with gzip.open(
                scalefactor_path,
                "rt",
                encoding="utf-8"
            ) as handle:
                scale_factors = json.load(handle)

        else:
            with open(
                scalefactor_path,
                "r",
                encoding="utf-8"
            ) as handle:
                scale_factors = json.load(handle)

        adata.uns["spatial"][sample_id]["scalefactors"] = (
            scale_factors
        )

    return adata

In [ ]:
# ============================================================
# 4. Load all samples
# ============================================================

adatas = []
loaded_sample_names = []

for h5_source in h5_files:

    gsm, sample_name = extract_sample_information(
        h5_source.name
    )

    adata_sample = read_gse208253_sample(
        h5_source=h5_source,
        data_dir=data_dir,
        cache_dir=cache_dir,
        tissue_only=True
    )

    adatas.append(adata_sample)
    loaded_sample_names.append(sample_name)

    print(
        f"{gsm} | {sample_name} | "
        f"{adata_sample.n_obs} spots × "
        f"{adata_sample.n_vars} genes"
    )


# ============================================================
# 5. Concatenate
# ============================================================

adata_spatial_oral = ad.concat(
    adatas,
    join="outer",
    merge="same",
    label="batch",
    keys=loaded_sample_names,
    index_unique=None
)

print("\nCombined oral-cancer spatial object:")
print(adata_spatial_oral)

print("\nSpots by sample:")
print(
    adata_spatial_oral.obs["sample"]
    .value_counts()
    .sort_index(
        key=lambda x: x.str.extract(r"(\d+)")[0].astype(int)
    )
)

In [ ]:
adata_spatial_oral

In [ ]:
print(adata_spatial_oral.obsm.keys())
print(adata_spatial_oral.obsm["spatial"][:5])
print(adata_spatial_oral.obs[["sample", "condition"]].head())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Choose sample by position
ad_plot = adatas[1]

sample_name = ad_plot.obs["sample"].iloc[0]
gsm = ad_plot.obs["gsm"].iloc[0]
sample_id = ad_plot.obs["sample_id"].iloc[0]

print("Plotting sample:", sample_name)
print("GEO sample:", gsm)
print("Sample ID:", sample_id)
print("Number of spots:", ad_plot.n_obs)

coords = np.asarray(ad_plot.obsm["spatial"])

plt.figure(figsize=(7, 7))

plt.scatter(
    coords[:, 0],
    coords[:, 1],
    s=5
)

plt.gca().invert_yaxis()
plt.gca().set_aspect("equal")

plt.xlabel("Spatial x-coordinate")
plt.ylabel("Spatial y-coordinate")
plt.title(
    f"GSE208253 oral cancer: {sample_id}"
)

plt.tight_layout()
plt.show()

In [ ]:
# Preprocess spatial data for scAICME-style semi-supervised clustering
import scanpy as sc
import numpy as np
import pandas as pd


# keep only tissue spots if available
if "in_tissue" in adata_spatial_oral.obs.columns:
    adata_spatial_oral = adata_spatial_oral[adata_spatial_oral.obs["in_tissue"].astype(int) == 1].copy()

adata=adata_spatial_oral
# Keep a raw copy before HVG restriction if needed later
print(f"START  | cells={adata.n_obs:,}  genes={adata.n_vars:,}")
adata.var_names = adata.var_names.astype(str)
adata.var_names_make_unique()
vn_up = adata.var_names.str.upper()

# --- Define QC gene sets and SAVE as columns in adata.var ---
# Human and mouse 'mt-' both become 'MT-' after uppercasing
adata.var["mt"]   = vn_up.str.startswith("MT-")
adata.var["ribo"] = vn_up.str.startswith(("RPS","RPL","MRPS","MRPL"))
adata.var["hb"]   = adata.var_names.str.match(r"^(HB[ABEDM][A-Z0-9]*)", case=False)

# --- Compute QC metrics using those var columns ---
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt","ribo","hb"],   # <- names of columns in adata.var
    percent_top=None,
    log1p=False,
    inplace=True
)

print(adata.obs[["n_genes_by_counts","total_counts","pct_counts_mt"]].describe())

# --- Choose thresholds (data-driven) ---
min_genes = 200
min_cells_gene = 3
umi_hi   = np.percentile(adata.obs["total_counts"], 99.5)
genes_hi = np.percentile(adata.obs["n_genes_by_counts"], 99.5)
mito_hi  = max(20.0, np.percentile(adata.obs["pct_counts_mt"], 95))

print(f"Thresholds → min_genes={min_genes}, umi_hi≈{umi_hi:.0f}, genes_hi≈{genes_hi:.0f}, mito_hi≈{mito_hi:.1f}%")

keep_cells = (
    (adata.obs["n_genes_by_counts"] >= min_genes) &
    (adata.obs["n_genes_by_counts"] <= genes_hi) &
    (adata.obs["total_counts"]      <= umi_hi) &
    (adata.obs["pct_counts_mt"]     <= mito_hi)
)
before = adata.n_obs
adata = adata[keep_cells].copy()
print(f"CELL FILTER | kept {adata.n_obs:,}/{before:,} ({adata.n_obs/before:.1%})")

# --- Filter low-prevalence genes ---
before_g = adata.n_vars
sc.pp.filter_genes(adata, min_cells=min_cells_gene)
print(f"GENE FILTER | kept {adata.n_vars:,}/{before_g:,} (≥{min_cells_gene} cells)")

# --- Normalize & log1p ---
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata
print(f"READY | cells={adata.n_obs:,} genes={adata.n_vars:,}")


In [ ]:
OSCC_ORAL_MARKERS = {

    # --------------------------------------------------------
    # B cell
    # --------------------------------------------------------
    "B_cell": [
        "CD79A",
        "MS4A1",
        "CD37",
        "CD74",
        "HLA-DRA",
        "CD22"
    ],

    # --------------------------------------------------------
    # Cancer cell
    # --------------------------------------------------------
    "Cancer_cell": [
        "EPCAM",
        "KRT8",
        "KRT18",
        "KRT19",
        "KRT14",
        "KRT17"
    ],

    # --------------------------------------------------------
    # Cytotoxic CD8+ T
    # --------------------------------------------------------
    "Cytotoxic_CD8_T": [
        "CD3D",
        "CD3E",
        "CD8A",
        "CD8B",
        "NKG7",
        "CCL5",
        "GZMK",
        "GZMB"
    ],


    # --------------------------------------------------------
    # Dendritic
    # --------------------------------------------------------
    "Dendritic": [
        "CD1C",
        "FCER1A",
        "CLEC10A",
        "HLA-DRA",
        "HLA-DPA1",
        "CST3"
    ],

    # --------------------------------------------------------
    # Detox-iCAF
    # --------------------------------------------------------
    "Detox_iCAF": [
        "ADH1B",
        "GPX3",
        "CFD",
        "APOD",
        "DCN",
        "LUM"
    ],

    # --------------------------------------------------------
    # ECM-myCAF
    # --------------------------------------------------------
    "ECM_myCAF": [
        "LRRC15",
        "GJB2",
        "COL1A1",
        "COL1A2",
        "COL3A1",
        "POSTN",
        "FN1",
        "THBS2"
    ],

    # --------------------------------------------------------
    # Endothelial
    # --------------------------------------------------------
    "Endothelial": [
        "PECAM1",
        "VWF",
        "CDH5",
        "CLDN5",
        "KDR",
        "EMCN"
    ],

    # --------------------------------------------------------
    # Intermediate fibroblast
    # --------------------------------------------------------
    "Intermediate_fibroblast": [
        "CXCL1",
        "PDPN",
        "DCN",
        "LUM",
        "COL1A1",
        "COL3A1"
    ],

    # --------------------------------------------------------
    # Macrophage
    # --------------------------------------------------------
    "Macrophage": [
        "C1QA",
        "C1QB",
        "C1QC",
        "APOE",
        "CD68",
        "FCGR3A",
        "CTSD"
    ],

    # --------------------------------------------------------
    # Mast
    # --------------------------------------------------------
    "Mast": [
        "TPSAB1",
        "TPSB2",
        "CPA3",
        "KIT",
        "MS4A2",
        "HDC"
    ],

    # --------------------------------------------------------
    # Myofibroblast
    # --------------------------------------------------------
    "Myofibroblast": [
        "ACTA2",
        "TAGLN",
        "MYL9",
        "TPM2",
        "CALD1",
        "CNN1"
    ],

    # --------------------------------------------------------
    # T regulatory cells
    # --------------------------------------------------------
    "Treg": [
        "FOXP3",
        "IL2RA",
        "CTLA4",
        "TIGIT",
        "IKZF2",
        "TNFRSF18"
    ]
}

In [ ]:
# Method 1: adaptive marker-based weak seed labeling on spatial transcriptomics spots
import scanpy as sc
import numpy as np
import pandas as pd

def adaptive_marker_threshold(adata, gene, quantile=0.75):
    vals = adata[:, gene].X
    if hasattr(vals, "toarray"):
        vals = vals.toarray()
    vals = np.asarray(vals).ravel()
    vals = vals[vals > 0]
    if vals.size == 0:
        return np.inf
    return float(np.quantile(vals, quantile))

def weak_label_quota_with_min_cells(
    adata,
    marker_sets,
    gene_quantile=0.75,
    target_frac=0.50,
    min_cells_per_type=20,
    min_score=0.2,
    verbose=True
):
    N = adata.n_obs
    target_total = int(round(target_frac * N))
    scores = {}
    for cell_type, genes in marker_sets.items():
        genes = [g for g in genes if g in adata.var_names]
        if len(genes) == 0:
            continue
        thr = {g: adaptive_marker_threshold(adata, g, quantile=gene_quantile) for g in genes}
        M = sc.get.obs_df(adata, keys=list(thr.keys()))
        expr_high = M.gt(pd.Series(thr))
        frac_high = expr_high.sum(axis=1) / len(thr)
        scores[cell_type] = frac_high.astype(float)

    if len(scores) == 0:
        raise ValueError("No marker genes found in adata.var_names")

    score_df = pd.DataFrame(scores, index=adata.obs_names)
    best_score = score_df.max(axis=1).astype(float)
    best_type  = score_df.idxmax(axis=1).astype(str)

    eligible_mask = best_score >= float(min_score)
    eligible_types_counts = best_type[eligible_mask].value_counts()
    keep_types = eligible_types_counts[eligible_types_counts >= min_cells_per_type].index.tolist()

    if len(keep_types) == 0:
        adata.obs["weak_label"] = pd.Categorical(["unlabeled"] * N)
        adata.obs["weak_conf"]  = best_score.values
        return adata, score_df

    max_types_possible = max(1, target_total // max(1, min_cells_per_type))
    if len(keep_types) > max_types_possible:
        keep_types = eligible_types_counts.loc[keep_types].sort_values(ascending=False).index[:max_types_possible].tolist()

    base = {t: min_cells_per_type for t in keep_types}
    base_total = sum(base.values())
    remaining = max(0, target_total - base_total)
    avail = eligible_types_counts.loc[keep_types].to_dict()
    extra = {t: 0 for t in keep_types}

    if remaining > 0:
        weights = np.array([max(0, avail[t] - base[t]) for t in keep_types], dtype=float)
        if weights.sum() > 0:
            props = weights / weights.sum()
            raw_extra = np.floor(props * remaining).astype(int)
            for t, e in zip(keep_types, raw_extra):
                extra[t] = int(e)
            leftover = remaining - sum(extra.values())
            if leftover > 0:
                order = np.argsort(-props)
                k = 0
                while leftover > 0:
                    t = keep_types[order[k % len(order)]]
                    extra[t] += 1
                    leftover -= 1
                    k += 1

    quota = {t: min(base[t] + extra[t], avail[t]) for t in keep_types}

    weak_label = np.array(["unlabeled"] * N, dtype=object)
    best_type_arr = best_type.values
    best_score_arr = best_score.values
    eligible_idx = np.where(eligible_mask.values)[0]

    for t in keep_types:
        idx_t = eligible_idx[best_type_arr[eligible_idx] == t]
        idx_sorted = idx_t[np.argsort(-best_score_arr[idx_t])]
        pick = idx_sorted[:quota[t]]
        weak_label[pick] = t

    adata.obs["weak_label"] = pd.Categorical(weak_label)
    adata.obs["weak_conf"]  = best_score.values

    if verbose:
        print(adata.obs["weak_label"].value_counts())
        print(f"Labeled fraction: {(adata.obs['weak_label'] != 'unlabeled').mean():.2%}")

    return adata, score_df

adata, marker_score_df = weak_label_quota_with_min_cells(
    adata,
    OSCC_ORAL_MARKERS,
    gene_quantile=0.75,
    target_frac=0.60,
    min_cells_per_type=10,
    min_score=0.20,
    verbose=True
)


#Method3

import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

def row_softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    E = np.exp(Z)
    return E / (E.sum(axis=1, keepdims=True) + 1e-12)

def gcn_seed_labeling_quota(
    adata,
    score_df,                         # cells x types marker_score_df
    target_frac=0.40,                 # desired final labeled fraction
    temperature=0.25,                 # 0.15–0.5 (lower = sharper)
    alpha=0.65,                       # 0.5–0.8 (lower = less oversmoothing)
    n_iter=50,
    tol=1e-4,                         # early stop
    # adaptive gates (quantiles)
    q_score=0.40,                     # gate on top1 >= quantile(top1)
    q_margin=0.35,                    # gate on margin >= quantile(margin)
    # quota controls
    min_cells_per_type=100,           # ensure each type has >= this many seeds (if possible)
    # minimum-size filter after quota (keep rare types!)
    min_cells_per_type_pct=0.001,     # 0.1% of n (≈68 for 68k)
    min_cells_floor=30,               # absolute min
    # graph construction
    neighbors_k=30,
    pca_comps=50,
    use_rep="X_pca",
    label_col="weak_label_gcn",
    conf_col="seed_conf_gcn",
    verbose=True
):
    # ---------- 0) Align score_df ----------
    if not isinstance(score_df, pd.DataFrame):
        raise ValueError("score_df must be a pandas DataFrame (cells x types).")

    score_df = score_df.copy()
    score_df.index = score_df.index.astype(str)
    score_df = score_df.reindex(adata.obs_names.astype(str))
    score_df = score_df.loc[:, score_df.notna().any(axis=0)]
    if score_df.shape[1] == 0:
        raise ValueError("score_df has no usable type columns after filtering.")

    types = score_df.columns.astype(str).tolist()
    S0 = score_df.fillna(0.0).to_numpy(dtype=float)

    # ---------- 1) Build sharpened probability seeds ----------
    # softmax produces probability-like vectors; temperature sharpens
    Y0 = row_softmax(S0 / float(temperature))

    # ---------- 2) Ensure PCA + neighbors ----------
    if (use_rep not in adata.obsm) or (adata.obsm[use_rep] is None):
        sc.pp.pca(adata, n_comps=pca_comps, svd_solver="arpack")
        use_rep = "X_pca"

    if "connectivities" not in adata.obsp:
        sc.pp.neighbors(
            adata,
            n_neighbors=neighbors_k,
            n_pcs=min(pca_comps, adata.obsm[use_rep].shape[1])
        )

    A = adata.obsp["connectivities"]
    if not sparse.isspmatrix(A):
        A = sparse.csr_matrix(A)

    N = adata.n_obs
    I = sparse.eye(N, format="csr")
    A_tilde = (A + I).tocsr()

    deg = np.asarray(A_tilde.sum(axis=1)).ravel()
    deg_inv_sqrt = 1.0 / np.sqrt(deg + 1e-12)
    D_inv_sqrt = sparse.diags(deg_inv_sqrt, format="csr")
    A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt

    # ---------- 3) Propagate with early stop ----------
    Y = Y0.copy()
    for it in range(int(n_iter)):
        Y_new = alpha * (A_hat @ Y) + (1.0 - alpha) * Y0
        Y_new = np.asarray(Y_new)
        diff = float(np.mean(np.abs(Y_new - Y)))
        Y = Y_new
        if diff < tol:
            if verbose:
                print(f"[GCN] early stop at iter={it+1}, mean|Δ|={diff:.2e}")
            break

    # ---------- 4) Compute top1/top2/margin ----------
    top1_idx = Y.argmax(axis=1)
    top1 = Y[np.arange(N), top1_idx]

    Y2 = Y.copy()
    Y2[np.arange(N), top1_idx] = -np.inf
    top2 = Y2.max(axis=1)
    margin = top1 - top2

    # ---------- 5) Adaptive gate (key fix vs fixed delta=0.15) ----------
    min_score_eff = float(np.quantile(top1, q_score))
    delta_eff = float(np.quantile(margin, q_margin))

    gate = (top1 >= min_score_eff) & (margin >= delta_eff)

    if verbose:
        print(f"[GCN] adaptive gate: min_score_eff={min_score_eff:.3f}, "
              f"delta_eff={delta_eff:.3f}, pass={gate.mean():.2%}")
        print(f"[GCN] median top1={np.median(top1):.3f}, median margin={np.median(margin):.3f}")

    # preliminary labels + conf
    pred_type = np.array(types, dtype=object)[top1_idx]
    conf = top1.copy()

    # ---------- 6) QUOTA selection to reach target_frac and keep many types ----------
    target_n = int(round(float(target_frac) * N))
    idx_gate = np.where(gate)[0]

    labels = np.array(["unlabeled"] * N, dtype=object)
    seed_conf = np.zeros(N, dtype=float)

    if idx_gate.size == 0 or target_n == 0:
        adata.obs[label_col] = pd.Categorical(labels)
        adata.obs[conf_col] = seed_conf
        return {"Y_propagated": Y, "top1": top1, "top2": top2, "margin": margin, "types": types}

    # group gated indices by predicted type
    gate_df = pd.DataFrame({
        "idx": idx_gate,
        "type": pred_type[idx_gate],
        "conf": conf[idx_gate]
    })

    # ensure each type gets min_cells_per_type if possible
    picked = []
    per_type_min = int(min_cells_per_type)

    for t in types:
        sub = gate_df[gate_df["type"] == t].sort_values("conf", ascending=False)
        if sub.shape[0] == 0:
            continue
        take = min(per_type_min, sub.shape[0])
        picked.extend(sub["idx"].values[:take].tolist())

    picked = list(dict.fromkeys(picked))  # unique preserve order
    remaining_budget = max(0, target_n - len(picked))

    if remaining_budget > 0:
        # fill remaining with highest confidence gated cells not yet picked
        rest = gate_df[~gate_df["idx"].isin(picked)].sort_values("conf", ascending=False)
        picked.extend(rest["idx"].values[:remaining_budget].tolist())

    # assign picked
    labels[picked] = pred_type[picked]
    seed_conf[picked] = conf[picked]

    # ---------- 7) Drop very small types (looser than your pct=0.005) ----------
    min_cells_pct = int(np.floor(float(min_cells_per_type_pct) * N))
    min_cells = max(int(min_cells_floor), int(min_cells_pct), 5)

    vc = pd.Series(labels).value_counts()
    keep_types = set(vc[vc >= min_cells].index.tolist())
    keep_types.discard("unlabeled")

    small_mask = (labels != "unlabeled") & (~pd.Series(labels).isin(keep_types).values)
    labels[small_mask] = "unlabeled"
    seed_conf[small_mask] = 0.0

    # ---------- 8) Write back ----------
    adata.obs[label_col] = pd.Categorical(labels)
    if "unlabeled" not in adata.obs[label_col].cat.categories:
        adata.obs[label_col] = adata.obs[label_col].cat.add_categories(["unlabeled"])
    adata.obs[label_col] = adata.obs[label_col].fillna("unlabeled")
    adata.obs[conf_col] = seed_conf

    if verbose:
        print(f"[GCN] min_cells_per_type used={min_cells}")
        print("[GCN] final label distribution:")
        print(adata.obs[label_col].value_counts())
        print(f"[GCN] labeled fraction={(adata.obs[label_col] != 'unlabeled').mean():.2%}")

    return {"Y_propagated": Y, "top1": top1, "top2": top2, "margin": margin, "types": types}



diag = gcn_seed_labeling_quota(
    adata,
    score_df=marker_score_df,

    # labeling quota
    target_frac=0.6,        # was 0.4 → allow more labeled cells

    # propagation
    temperature=0.1,        # was 0.1 → softer probability distribution
    alpha=0.5,              # was 0.5 → stronger graph propagation
    n_iter=50,              # was 50 → more smoothing

    # graph construction
    neighbors_k=50,          # was 50 → smoother graph
    pca_comps=50,

    label_col="weak_label",
    conf_col="seed_conf",

    verbose=True
)


In [ ]:
#Method2
import numpy as np
import pandas as pd
from sklearn.mixture import BayesianGaussianMixture
from sklearn.preprocessing import StandardScaler

def dp_seed_by_marker_sets_soft(
    adata,
    marker_sets,
    n_components=30,
    weight_concentration_prior=0.05,
    random_state=0,
    per_gene_pos_quantile=0.70,
    cluster_score_min=0.20,
    min_cells_cluster=100,
    label_col_prefix="dp_seed",
    conf_col_prefix="dp_seed_conf",
    verbose=True
):
    diagnostics = {}
    all_vars = set(adata.var_names.astype(str))
    n_cells = adata.n_obs

    # --- Pre-filter thresholds ---
    min_genes_present     = 3
    min_expressed_markers = 2
    min_cells_per_gene    = 20
    min_cell_enrichment   = 0.05

    for setname, genes in marker_sets.items():
        # 0) Keep only markers present in adata
        genes = [g for g in genes if g in all_vars]
        if len(genes) < min_genes_present:
            if verbose:
                print(f"[DP-soft][skip] {setname}: only {len(genes)} markers present in adata.")
            continue

        # 1) Extract expression
        Xg = adata[:, genes].X
        if hasattr(Xg, "toarray"):
            Xg = Xg.toarray()
        Xg = np.asarray(Xg, dtype=float)

        # 2) Pre-filter 1: expressed markers
        expr_per_gene = (Xg > 0).sum(axis=0)
        expressed_markers = int(np.sum(expr_per_gene >= min_cells_per_gene))
        if expressed_markers < min_expressed_markers:
            if verbose:
                print(f"[DP-soft][skip] {setname}: only {expressed_markers} expressed markers.")
            continue

        # 3) Pre-filter 2: cell enrichment
        cells_with_any_marker = (Xg > 0).sum(axis=1) > 0
        cell_enrichment = float(np.mean(cells_with_any_marker))
        if cell_enrichment < min_cell_enrichment:
            if verbose:
                print(f"[DP-soft][skip] {setname}: cell enrichment {cell_enrichment:.4f} < {min_cell_enrichment}.")
            continue

        # 4) Per-gene thresholds
        gene_thr = {}
        for j, g in enumerate(genes):
            v = Xg[:, j]
            pos = v[v > 0]
            if pos.size >= 20:
                gene_thr[g] = float(np.quantile(pos, per_gene_pos_quantile))
            elif pos.size > 0:
                gene_thr[g] = float(np.median(pos))
            else:
                gene_thr[g] = np.inf

        # 5) Cell marker scores
        high_mask = np.zeros_like(Xg, dtype=bool)
        for j, g in enumerate(genes):
            thr = gene_thr[g]
            if np.isinf(thr):
                continue
            high_mask[:, j] = Xg[:, j] > thr

        marker_score = high_mask.sum(axis=1) / max(1, len(genes))

        # 6) Standardize + BGM (NO PCA)
        scaler = StandardScaler(with_mean=True, with_std=True)
        Xg_scaled = scaler.fit_transform(Xg)

        bgm = BayesianGaussianMixture(
            n_components=n_components,
            weight_concentration_prior_type="dirichlet_process",
            weight_concentration_prior=weight_concentration_prior,
            covariance_type="full",
            max_iter=1000,
            n_init=1,
            random_state=random_state,
        )
        bgm.fit(Xg_scaled)
        cl = bgm.predict(Xg_scaled)

        # 7) Cluster-level stats
        df_cluster = pd.DataFrame({"cluster": cl, "score": marker_score})
        cluster_summary = df_cluster.groupby("cluster")["score"].agg(["mean", "count"])

        good_clusters = cluster_summary[
            (cluster_summary["mean"] >= cluster_score_min) &
            (cluster_summary["count"] >= min_cells_cluster)
        ].index.tolist()

        # 8) Assign labels + confidence
        label_col = f"{label_col_prefix}_{setname}"
        conf_col  = f"{conf_col_prefix}_{setname}"

        labels = np.array(["unlabeled"] * n_cells, dtype=object)
        confs  = np.zeros(n_cells, dtype=float)

        if len(good_clusters) > 0:
            max_score = marker_score.max() if marker_score.max() > 0 else 1.0
            for cid in good_clusters:
                idx = np.where(cl == cid)[0]
                labels[idx] = setname
                confs[idx] = marker_score[idx] / max_score

        adata.obs[label_col] = pd.Categorical(labels)
        adata.obs[conf_col]  = confs

        # 9) Diagnostics
        n_lab = (labels != "unlabeled").sum()
        pct_lab = n_lab / n_cells * 100.0
        diagnostics[setname] = {
            "n_markers_used": len(genes),
            "n_clusters": int(len(cluster_summary)),
            "good_clusters": good_clusters,
            "n_labelled": int(n_lab),
            "pct_labelled": float(pct_lab),
            "cluster_summary": cluster_summary,
            "expressed_markers": expressed_markers,
            "cell_enrichment": cell_enrichment,
            "n_features_used": int(Xg_scaled.shape[1]),
        }

        if verbose:
            print(
                f"{setname}: labelled {n_lab}/{n_cells} ({pct_lab:.2f}%), "
                f"good_clusters={len(good_clusters)}, n_features={Xg_scaled.shape[1]}"
            )

    # --- Build consensus weak label across marker sets ---
    label_cols = [f"{label_col_prefix}_{s}" for s in marker_sets.keys() if f"{label_col_prefix}_{s}" in adata.obs]
    conf_cols  = [f"{conf_col_prefix}_{s}" for s in marker_sets.keys() if f"{conf_col_prefix}_{s}" in adata.obs]

    if len(label_cols) > 0:
        lab_df = adata.obs[label_cols].astype(str)
        conf_df = adata.obs[conf_cols].astype(float).fillna(0.0)

        dp_consensus_label = []
        dp_consensus_conf  = []

        for i in adata.obs_names:
            row_labels = lab_df.loc[i].values
            row_confs  = conf_df.loc[i].values
            row_confs = np.where(row_labels == "unlabeled", 0.0, row_confs)

            best_idx = int(np.argmax(row_confs))
            best_conf = float(row_confs[best_idx])
            best_label = row_labels[best_idx] if best_conf > 0 else "unlabeled"

            dp_consensus_label.append(best_label)
            dp_consensus_conf.append(best_conf)

        seed_labels = pd.Categorical(dp_consensus_label)
        if "unlabeled" not in seed_labels.categories:
            seed_labels = seed_labels.add_categories(["unlabeled"])
        adata.obs["weak_label"] = seed_labels
        adata.obs["weak_conf"]  = np.array(dp_consensus_conf, dtype=float)

        # --- Final size filter ---
        #threshold = max(100, int(0.002 * adata.n_obs))
        threshold = max(20, int(0.001 * adata.n_obs))
        counts = adata.obs["weak_label"].value_counts()

        keep_types = set(counts[counts >= threshold].index.tolist())

        mask_keep = adata.obs["weak_label"].isin(keep_types)
        adata.obs.loc[~mask_keep, "weak_label"] = "unlabeled"
        adata.obs.loc[~mask_keep, "weak_conf"]  = 0.0

        if verbose:
            print("\n[DP-soft no PCA] Final weak_label distribution:")
            print(adata.obs["weak_label"].value_counts(normalize=True) * 100)

    return diagnostics

In [ ]:
N = adata.n_obs
#n_components = min(15, int(np.sqrt(N))) # capped adaptive choice
n_components = 15

diag = dp_seed_by_marker_sets_soft(
    adata,
    OSCC_ORAL_MARKERS,                    # PCA dimension for each marker block
    per_gene_pos_quantile=0.3,
    cluster_score_min=0.08,
    min_cells_cluster=30,
    n_components=n_components,
    weight_concentration_prior=0.1,
    random_state=42
)

for k, v in diag.items():
    print(f"{k}: labelled {v['n_labelled']} cells ({v['pct_labelled']:.2f}%)")

In [ ]:
adata.obs["weak_label"]

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from collections import Counter
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

# -----------------------------
# PCA / neighbors
# -----------------------------
if "X_pca" not in adata.obsm:
    sc.pp.pca(adata, n_comps=20, svd_solver="arpack")

n_pcs_use = min(15, adata.obsm["X_pca"].shape[1])

if "connectivities" not in adata.obsp:
    sc.pp.neighbors(adata, n_neighbors=20, n_pcs=n_pcs_use)
# -----------------------------
# Helper
# -----------------------------
def get_training_data(
    adata,
    X_key="X_pca",
    seed_col="weak_label",
    conf_col="weak_conf",
    min_seed_conf=0.05,
    max_pcs=15
):
    X = np.asarray(adata.obsm[X_key], dtype=np.float32)
    X = X[:, :min(max_pcs, X.shape[1])]
    y = adata.obs[seed_col].astype(str).values

    if conf_col in adata.obs.columns:
        conf = np.asarray(adata.obs[conf_col], dtype=float)
        known_mask = (y != "unlabeled") & (conf >= min_seed_conf)
    else:
        known_mask = (y != "unlabeled")

    y_known = y[known_mask]
    classes = np.unique(y_known)

    print("Known labeled spots:", known_mask.sum())
    print("Known classes:", classes.tolist())

    if known_mask.sum() < 20:
        raise ValueError(f"Too few confident labeled seeds: {known_mask.sum()}")
    if len(classes) < 2:
        raise ValueError(f"Need at least 2 labeled classes, got {len(classes)}: {classes.tolist()}")

    return X, y, known_mask

# -----------------------------
# Methods
# -----------------------------
def rbf_svm_labeling(adata, X_key="X_pca", seed_col="weak_label", conf_col="weak_conf",
                     out_col="label_svm_rbf", out_conf_col="svm_rbf_conf",
                     C=2.0, gamma="scale", min_seed_conf=0.05, min_conf=0.3, max_pcs=15):
    X, y, known_mask = get_training_data(adata, X_key, seed_col, conf_col, min_seed_conf, max_pcs)
    clf = make_pipeline(
        StandardScaler(),
        SVC(C=C, gamma=gamma, kernel="rbf", probability=True, class_weight="balanced", random_state=42)
    )
    clf.fit(X[known_mask], y[known_mask])
    proba = clf.predict_proba(X)
    pred = clf.classes_[np.argmax(proba, axis=1)]
    conf = np.max(proba, axis=1)
    y_out = pred.copy()
    y_out[conf < min_conf] = "unlabeled"
    adata.obs[out_col] = pd.Categorical(y_out)
    adata.obs[out_conf_col] = conf

def kmeans_seed_transfer(adata, X_key="X_pca", seed_col="weak_label", conf_col="weak_conf",
                         out_col="label_kmeans", out_conf_col="kmeans_conf", min_seed_conf=0.05):
    X, y, known_mask = get_training_data(adata, X_key, seed_col, conf_col, min_seed_conf, 30)
    k = min(10, max(8, int(np.sqrt(adata.n_obs / 2))))
    km = KMeans(n_clusters=k, random_state=42, n_init=20, max_iter=500)
    km.fit(X)
    known_types = np.unique(y[known_mask])
    centroids = np.vstack([X[known_mask & (y == t)].mean(axis=0) for t in known_types])

    cluster_to_type, cluster_conf = {}, {}
    for c in np.unique(km.labels_):
        idx = np.where(km.labels_ == c)[0]
        seed_in_c = y[idx]
        seed_in_c = seed_in_c[seed_in_c != "unlabeled"]
        if len(seed_in_c) > 0:
            counts = Counter(seed_in_c)
            maj_label, maj_count = counts.most_common(1)[0]
            cluster_to_type[c] = maj_label
            cluster_conf[c] = maj_count / len(idx)
        else:
            nearest = pairwise_distances(km.cluster_centers_[c].reshape(1, -1), centroids).argmin()
            cluster_to_type[c] = known_types[nearest]
            cluster_conf[c] = 0.0

    labels_out = np.array([cluster_to_type[c] for c in km.labels_], dtype=object)
    conf_out = np.array([cluster_conf[c] for c in km.labels_], dtype=float)
    adata.obs[out_col] = pd.Categorical(labels_out)
    adata.obs[out_conf_col] = conf_out

def knn_labeling(adata, X_key="X_pca", seed_col="weak_label", conf_col="weak_conf",
                 out_col="label_knn", out_conf_col="knn_conf",
                 n_neighbors=9, min_seed_conf=0.05, min_conf=0.3, max_pcs=15):
    X, y, known_mask = get_training_data(adata, X_key, seed_col, conf_col, min_seed_conf, max_pcs)
    k = min(n_neighbors, max(1, known_mask.sum() - 1))
    knn = KNeighborsClassifier(n_neighbors=k, weights="distance", p=2)
    knn.fit(X[known_mask], y[known_mask])
    pred = knn.predict(X)
    proba = knn.predict_proba(X)
    conf = np.max(proba, axis=1)
    y_out = pred.copy()
    y_out[conf < min_conf] = "unlabeled"
    adata.obs[out_col] = pd.Categorical(y_out)
    adata.obs[out_conf_col] = conf

def rf_labeling(adata, X_key="X_pca", seed_col="weak_label", conf_col="weak_conf",
                out_col="label_rf", out_conf_col="rf_conf",
                n_estimators=300, max_depth=18, min_samples_leaf=3,
                min_seed_conf=0.05, min_conf=0.3, max_pcs=15):
    X, y, known_mask = get_training_data(adata, X_key, seed_col, conf_col, min_seed_conf, max_pcs)
    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        max_features="sqrt",
        n_jobs=-1,
        class_weight="balanced_subsample",
        random_state=42
    )
    rf.fit(X[known_mask], y[known_mask])
    pred = rf.predict(X)
    proba = rf.predict_proba(X)
    conf = np.max(proba, axis=1)
    y_out = pred.copy()
    y_out[conf < min_conf] = "unlabeled"
    adata.obs[out_col] = pd.Categorical(y_out)
    adata.obs[out_conf_col] = conf

def mlp_labeling(adata, X_key="X_pca", seed_col="weak_label", conf_col="weak_conf",
                 out_col="label_mlp", out_conf_col="mlp_conf",
                 hidden_layer_sizes=(128, 64), alpha=1e-3, max_iter=300,
                 min_seed_conf=0.05, min_conf=0.3, max_pcs=15):
    X, y_str, known_mask = get_training_data(adata, X_key, seed_col, conf_col, min_seed_conf, max_pcs)
    le = LabelEncoder()
    y_known = le.fit_transform(y_str[known_mask])

    clf = make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            activation="relu",
            solver="adam",
            alpha=alpha,
            learning_rate_init=1e-3,
            max_iter=max_iter,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=20,
            random_state=42
        )
    )
    clf.fit(X[known_mask], y_known)
    proba = clf.predict_proba(X)
    pred_int = np.argmax(proba, axis=1)
    pred_lbl = le.inverse_transform(pred_int)
    conf = np.max(proba, axis=1)
    y_out = pred_lbl.copy()
    y_out[conf < min_conf] = "unlabeled"
    adata.obs[out_col] = pd.Categorical(y_out)
    adata.obs[out_conf_col] = conf

# -----------------------------
# Run methods safely
# -----------------------------
method_outputs = []

for name, func in [
    ("label_svm_rbf", rbf_svm_labeling),
    ("label_kmeans", kmeans_seed_transfer),
    ("label_knn", knn_labeling),
    ("label_rf", rf_labeling),
    ("label_mlp", mlp_labeling),
]:
    try:
        print(f"\nRunning {name} ...")
        func(adata)
        method_outputs.append(name)
        print(f"{name} done")
    except Exception as e:
        print(f"{name} failed: {e}")

print("\nAvailable classifier outputs:", method_outputs)
print("Current obs columns:", adata.obs.columns.tolist())

# -----------------------------
# Consensus from available methods
# -----------------------------
if len(method_outputs) == 0:
    raise ValueError("No classifier output was generated. Check weak_label distribution.")

methods = np.vstack([
    adata.obs[c].astype(str).values
    for c in method_outputs
])

consensus = np.array(["unlabeled"] * adata.n_obs, dtype=object)
agree_frac = np.zeros(adata.n_obs)

for i in range(adata.n_obs):
    votes = [m[i] for m in methods]
    filtered = [v for v in votes if v != "unlabeled"]
    if len(filtered) == 0:
        consensus[i] = "unlabeled"
        agree_frac[i] = 0.0
    else:
        c = Counter(filtered)
        maj_label, maj_votes = c.most_common(1)[0]
        consensus[i] = maj_label
        agree_frac[i] = maj_votes / len(method_outputs)

adata.obs["label_consensus"] = pd.Categorical(consensus)
adata.obs["consensus_agree_frac"] = agree_frac

print("\nConsensus created successfully.")
print(adata.obs["label_consensus"].value_counts(dropna=False))

In [ ]:
adata

In [ ]:
scaicme_labels = adata.obs[
    ["sample", "barcode", "label_consensus"]
].copy()

scaicme_labels.to_csv(
    "scAICME_spatial_oral_labels.csv",
    index=True
)

print(scaicme_labels.head())

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=1.2, key_added="label_leiden")



In [ ]:
print(adata.obs["label_leiden"].value_counts())


# --------------------------------------------------
# Save Leiden labels
# --------------------------------------------------
leiden_output = adata.obs[
    ["sample", "barcode", "label_leiden"]
].copy()

leiden_output.to_csv(
    "Leiden_spatial_labels.csv",
    index=True
)

print("Saved: Leiden_spatial_labels.csv")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

# ============================================================
# GSE208253 Oral Squamous Cell Carcinoma
# Skin-style spatial plot:
# Original H&E vs scAICME consensus
# ============================================================

# ------------------------------------------------------------
# Sample information
# ------------------------------------------------------------
sample_info = {
    "s1":  "GSM6339631",
    "s2":  "GSM6339632",
    "s3":  "GSM6339633",
    "s4":  "GSM6339634",
    "s5":  "GSM6339635",
    "s6":  "GSM6339636",
    "s7":  "GSM6339637",
    "s8":  "GSM6339638",
    "s9":  "GSM6339639",
    "s10": "GSM6339640",
    "s11": "GSM6339641",
    "s12": "GSM6339642",
}

oral_samples = list(sample_info.keys())

# ------------------------------------------------------------
# Choose sample
#
# 0=s1 ... 11=s12
# ------------------------------------------------------------
s = 0

sample_name = oral_samples[s]
gsm = sample_info[sample_name]

print("Selected:", sample_name, gsm)

# ------------------------------------------------------------
# Original H&E image
# ------------------------------------------------------------
original_image_path = (
    f"{data_dir}/"
    f"{gsm}_{sample_name}_tissue_hires_image.png"
)

original_image = np.asarray(
    Image.open(original_image_path).convert("RGB")
)

print("H&E image shape:", original_image.shape)

# ------------------------------------------------------------
# Original individual spatial object
# ------------------------------------------------------------
ad_orig = next(
    x.copy()
    for x in adatas
    if x.obs["sample"].iloc[0] == sample_name
)

ad_plot = ad_orig.copy()

print("Original spots:", ad_orig.n_obs)

# ============================================================
# Transfer scAICME labels
# ============================================================

idx = (
    adata.obs["sample"]
    .astype(str)
    == sample_name
)

merged_sample = adata[idx].copy()

print(
    "Merged sample spots:",
    merged_sample.n_obs
)

# ------------------------------------------------------------
# Match by obs_names first
# ------------------------------------------------------------
common = ad_plot.obs_names.intersection(
    merged_sample.obs_names
)

print(
    "Matched spot IDs:",
    len(common),
    "/",
    ad_plot.n_obs
)

ad_plot.obs["label_consensus"] = pd.NA

ad_plot.obs.loc[
    common,
    "label_consensus"
] = (
    merged_sample.obs.loc[
        common,
        "label_consensus"
    ]
    .astype(str)
    .values
)

# ------------------------------------------------------------
# If obs-name matching unexpectedly fails,
# use raw barcode matching
# ------------------------------------------------------------
if len(common) == 0:

    print(
        "obs_names did not match; "
        "switching to barcode matching."
    )

    lab = merged_sample.obs[
        [
            "barcode",
            "label_consensus"
        ]
    ].copy()

    lab["barcode"] = (
        lab["barcode"]
        .astype(str)
    )

    lab = (
        lab
        .drop_duplicates("barcode")
        .set_index("barcode")
    )

    ad_plot.obs["label_consensus"] = (
        ad_plot.obs["barcode"]
        .astype(str)
        .map(
            lab["label_consensus"]
        )
    )

# clean labels
ad_plot.obs["label_consensus"] = (
    ad_plot.obs["label_consensus"]
    .astype("string")
    .str.strip()
)

ad_plot.obs.loc[
    ad_plot.obs["label_consensus"].isin(
        ["nan", "None", "<NA>"]
    ),
    "label_consensus"
] = pd.NA

# ============================================================
# Keep tissue spots
# ============================================================

if "in_tissue" in ad_orig.obs.columns:

    ad_orig = ad_orig[
        ad_orig.obs["in_tissue"]
        .astype(int)
        == 1
    ].copy()

if "in_tissue" in ad_plot.obs.columns:

    ad_plot = ad_plot[
        ad_plot.obs["in_tissue"]
        .astype(int)
        == 1
    ].copy()

# only labelled spots on right
ad_plot = ad_plot[
    ad_plot.obs[
        "label_consensus"
    ].notna()
].copy()

print(
    "Final labeled spots:",
    ad_plot.n_obs
)

print("\nCell-type distribution:")

print(
    ad_plot.obs[
        "label_consensus"
    ].value_counts()
)

# ============================================================
# High-contrast palette
# Similar visual character to the skin figure
# ============================================================

# ============================================================
# Updated high-contrast color palette
# ============================================================

celltype_colors = {
    # --- Tumor / epithelial ---
    "Malignant_epithelial": "#E41A1C",      # bright red
    "Tumor_Epithelial": "#E41A1C",
    "Epithelial_tumor": "#E41A1C",
    "Basal_squamous": "#FF7F00",            # orange
    "Differentiated_squamous": "#4DAF4A",   # green
    "Partial_EMT_leading_edge": "#984EA3",  # purple

    # --- Fibroblast / stromal ---
    "Fibroblast": "#33A02C",                # green
    "CAF": "#A65628",                       # brown
    "Fibroblast_CAF": "#A65628",            # brown
    "Myofibroblast": "#F781BF",             # pink
    "Myofibroblast_CAF": "#F781BF",
    "Inflammatory_CAF": "#B15928",          # darker brown
    "ECM_CAF": "#6A3D9A",                   # violet

    # --- Vascular ---
    "Endothelial": "#00BFC4",               # cyan/teal
    "Pericyte": "#1F78B4",                  # blue
    "Smooth_muscle": "#8C564B",             # brownish
    "SmoothMuscle": "#8C564B",

    # --- Immune ---
    "T_cell": "#377EB8",                    # blue
    "T_CELL": "#377EB8",
    "Cytotoxic_T_NK": "#08519C",            # dark blue
    "NK": "#08519C",                        # dark blue
    "NK_cell": "#08519C",
    "B_cell": "#FFD92F",                    # yellow
    "Plasma_cell": "#E6AB02",               # mustard
    "Myeloid": "#A6761D",                   # ochre
    "Macrophage": "#7B3294",                # purple
    "Dendritic": "#66C2A5",                 # sea green
    "Mast": "#C51B7D",                      # magenta

    # --- Other ---
    "Adipocyte": "#B2DF8A",                 # light green
    "Eccrine_gland": "#1F78B4",
    "Keratinocyte": "#E31A1C",
    "Melanocyte": "#6A3D9A",
    "Nerve": "#999999"
}
# ============================================================
# Find correct scale factor
# ============================================================

# GSE208253 objects may have manually constructed spatial
# metadata. Use stored scalefactor when available.
library_ids = list(
    ad_orig.uns.get(
        "spatial",
        {}
    ).keys()
)

scale_factor = None

if len(library_ids) > 0:

    library_id = library_ids[0]

    spatial_info = (
        ad_orig.uns[
            "spatial"
        ][library_id]
    )

    if "scalefactors" in spatial_info:

        scale_factor = (
            spatial_info[
                "scalefactors"
            ].get(
                "tissue_hires_scalef",
                None
            )
        )

print(
    "Hires scale factor:",
    scale_factor
)

# ============================================================
# Coordinates
# ============================================================

coords_orig = np.asarray(
    ad_orig.obsm["spatial"]
).copy()

coords_plot = np.asarray(
    ad_plot.obsm["spatial"]
).copy()

# ------------------------------------------------------------
# Scale full-resolution Visium coordinates to hires image
# ------------------------------------------------------------
if scale_factor is not None:

    coords_orig = (
        coords_orig
        * scale_factor
    )

    coords_plot = (
        coords_plot
        * scale_factor
    )

else:

    # --------------------------------------------------------
    # If no scalefactor is stored, estimate scaling from the
    # coordinate/image dimensions for visualization.
    # --------------------------------------------------------

    img_h, img_w = (
        original_image.shape[:2]
    )

    x_max = np.nanmax(
        coords_orig[:, 0]
    )

    y_max = np.nanmax(
        coords_orig[:, 1]
    )

    sx = img_w / x_max
    sy = img_h / y_max

    # retain aspect ratio
    estimated_scale = min(
        sx,
        sy
    )

    print(
        "Using estimated plotting scale:",
        estimated_scale
    )

    coords_orig *= estimated_scale
    coords_plot *= estimated_scale

# ============================================================
# Labels and categories
# ============================================================

labels = (
    ad_plot.obs[
        "label_consensus"
    ]
    .astype(str)
    .values
)

categories = sorted(
    pd.unique(labels)
)

# ------------------------------------------------------------
# Never make unknown labels black
# ------------------------------------------------------------
fallback_colors = [
    "#E41A1C",
    "#377EB8",
    "#4DAF4A",
    "#984EA3",
    "#FF7F00",
    "#A65628",
    "#F781BF",
    "#999999",
    "#66C2A5",
    "#FC8D62",
    "#8DA0CB",
    "#E78AC3",
]

for i, ct in enumerate(categories):

    if ct not in celltype_colors:

        celltype_colors[ct] = (
            fallback_colors[
                i % len(
                    fallback_colors
                )
            ]
        )

print("\nColor mapping:")

for ct in categories:

    print(
        f"{ct:30s}",
        celltype_colors[ct]
    )

# ============================================================
# FIGURE
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(16, 7),
    gridspec_kw={
        "wspace": 0.03
    }
)

# ============================================================
# LEFT: Original tissue
# ============================================================

axes[0].imshow(
    original_image
)

axes[0].set_title(
    "Original tissue",
    fontsize=15
)

axes[0].axis(
    "off"
)

# ============================================================
# RIGHT: scAICME consensus
# Same H&E background
# ============================================================

axes[1].imshow(
    original_image
)

for cell_type in categories:

    mask = (
        labels
        == cell_type
    )

    axes[1].scatter(
        coords_plot[
            mask,
            0
        ],
        coords_plot[
            mask,
            1
        ],

        # skin-like prominent spots
        s=38,

        color=(
            celltype_colors[
                cell_type
            ]
        ),

        label=cell_type,

        alpha=0.98,

        edgecolors="none",

        zorder=3
    )

axes[1].set_title(
    "scAICME consensus",
    fontsize=15
)

axes[1].axis(
    "off"
)

# ============================================================
# Force both panels to use identical image extent
# ============================================================

img_h, img_w = (
    original_image.shape[:2]
)

for ax in axes:

    ax.set_xlim(
        0,
        img_w
    )

    ax.set_ylim(
        img_h,
        0
    )

    ax.set_aspect(
        "equal"
    )

# ============================================================
# Legend
# ============================================================

axes[1].legend(
    bbox_to_anchor=(
        1.015,
        0.5
    ),
    loc="center left",
    frameon=False,
    fontsize=11,
    markerscale=1.2,
    borderaxespad=0.0
)

# ============================================================
# Overall title
# ============================================================

fig.suptitle(
    "GSE208253 Oral Squamous Cell Carcinoma",
    fontsize=17,
    y=0.98
)

# ============================================================
# Save
# ============================================================

plt.tight_layout(
    rect=[
        0,
        0,
        0.87,
        0.94
    ]
)

outfile = (
    f"{sample_name}_OSCC_"
    f"original_vs_scAICME_consensus.png"
)

plt.savefig(
    outfile,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "\nSaved:",
    outfile
)

In [ ]:
# ============================================================
# Create breast CAF subset
# ============================================================

caf_types = [

    # --------------------------------------------------------
    # B cell
    # --------------------------------------------------------
    "B_cell",

    # --------------------------------------------------------
    # Cancer cell
    # --------------------------------------------------------
    "Cancer_cell",

    # --------------------------------------------------------
    # Cytotoxic CD8+ T
    # --------------------------------------------------------
    "Cytotoxic_CD8_T",


    # --------------------------------------------------------
    # Dendritic
    # --------------------------------------------------------
    "Dendritic",

    # --------------------------------------------------------
    # Detox-iCAF
    # --------------------------------------------------------
    "Detox_iCAF",

    # --------------------------------------------------------
    # ECM-myCAF
    # --------------------------------------------------------
    "ECM_myCAF",

    # --------------------------------------------------------
    # Endothelial
    # --------------------------------------------------------
    "Endothelial",

    # --------------------------------------------------------
    # Intermediate fibroblast
    # --------------------------------------------------------
    "Intermediate_fibroblast",

    # --------------------------------------------------------
    # Macrophage
    # --------------------------------------------------------
    "Macrophage",

    # --------------------------------------------------------
    # Mast
    # --------------------------------------------------------
    "Mast",

    # --------------------------------------------------------
    # Myofibroblast
    # --------------------------------------------------------
    "Myofibroblast",
]

print("Available labels:")
print(
    adata.obs["label_consensus"]
    .value_counts()
)

adata_caf = adata[
    adata.obs["label_consensus"]
    .astype(str)
    .isin(caf_types)
].copy()

print(
    "\nCAF subset:",
    adata_caf.shape
)

print(
    "\nCAF label distribution:"
)

print(
    adata_caf.obs["label_consensus"]
    .value_counts()
)

In [ ]:
marker_dict_plot = {}

for ct, genes in OSCC_ORAL_MARKERS.items():

    genes_present = [
        g
        for g in genes
        if g in adata_caf.var_names
    ]

    if len(genes_present) > 0:
        marker_dict_plot[ct] = genes_present

print(marker_dict_plot)

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import pandas as pd

adata_caf.obs["label_consensus"] = pd.Categorical(
    adata_caf.obs["label_consensus"],
    categories=caf_types,
    ordered=True
)

# make matrixplot without internal title
mp = sc.pl.matrixplot(
    adata_caf,
    marker_dict_plot,
    groupby="label_consensus",
    standard_scale="var",
    cmap="RdBu_r",
    dendrogram=False,
    figsize=(18, 6),
    show=False,
    return_fig=True
)

# build figure first
mp.make_figure()

# add title manually with higher position
mp.fig.suptitle(
    "Oral Cancer Spatial Marker Gene Expression",
    fontsize=18,
    y=1.02
)

# give a bit more top margin
mp.fig.subplots_adjust(top=0.82)

mp.fig.savefig(
    "GSE208253_Oral_marker_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
if RUN_COMPARISONS:
    #Comparing metrics
    import numpy as np
    import pandas as pd

    from scipy import sparse
    from sklearn.neighbors import NearestNeighbors
    from sklearn.metrics import silhouette_score
    from sklearn.preprocessing import LabelEncoder


    # ============================================================
    # 1. Build spatial-neighbor graph WITHIN each tissue sample
    # ============================================================

    def build_samplewise_spatial_graph(
        adata,
        sample_col="sample",
        spatial_key="spatial",
        n_neighbors=6
    ):
        """
        Build a symmetric spatial kNN graph separately within each sample.
        Prevents spots from different tissue sections becoming neighbors.
        """

        if spatial_key not in adata.obsm:
            raise ValueError(f"{spatial_key!r} not found in adata.obsm")

        if sample_col not in adata.obs.columns:
            raise ValueError(f"{sample_col!r} not found in adata.obs")

        N = adata.n_obs

        rows = []
        cols = []
        vals = []

        samples = adata.obs[sample_col].astype(str).values

        for s in pd.unique(samples):

            idx = np.where(samples == s)[0]

            if len(idx) < 2:
                continue

            coords = np.asarray(
                adata.obsm[spatial_key][idx],
                dtype=float
            )

            # cannot request more neighbours than cells
            k = min(n_neighbors + 1, len(idx))

            nbrs = NearestNeighbors(
                n_neighbors=k,
                metric="euclidean"
            ).fit(coords)

            neigh = nbrs.kneighbors(
                coords,
                return_distance=False
            )

            for local_i in range(len(idx)):

                global_i = idx[local_i]

                # first neighbour is normally self
                for local_j in neigh[local_i, 1:]:

                    global_j = idx[local_j]

                    rows.append(global_i)
                    cols.append(global_j)
                    vals.append(1.0)

        W = sparse.csr_matrix(
            (vals, (rows, cols)),
            shape=(N, N)
        )

        # make graph symmetric
        W = W.maximum(W.T)

        # remove self edges
        W.setdiag(0)
        W.eliminate_zeros()

        return W

    def spatial_coherence(
        labels,
        W
    ):
        """
        Fraction of spatial edges connecting spots with the same label.

        Higher = better spatial coherence.
        """

        labels = np.asarray(labels).astype(str)

        row, col = W.nonzero()

        if len(row) == 0:
            return np.nan

        same = labels[row] == labels[col]

        coherence = np.mean(same)

        return float(coherence)


    def spatial_inconsistency(
        labels,
        W
    ):
        """
        Fraction of spatial edges connecting different labels.

        Lower = better.
        """

        coherence = spatial_coherence(labels, W)

        return 1.0 - coherence

    def rare_domain_recovery_proxy(
        labels,
        W,
        rare_fraction=0.02,
        min_cells=3,
        exclude_labels=("unlabeled",)
    ):
        """
        Reference-free rare-domain recovery proxy.

        Rare type:
            <= rare_fraction of all labelled spots

        A rare spot is considered spatially recovered if
        at least one spatial neighbour has the same label.

        Higher = better.
        """

        labels = np.asarray(labels).astype(str)

        valid_mask = ~np.isin(labels, exclude_labels)
        valid_labels = labels[valid_mask]

        counts = pd.Series(valid_labels).value_counts()

        N = len(valid_labels)

        rare_types = counts[
            (counts <= rare_fraction * N) &
            (counts >= min_cells)
        ].index.tolist()

        if len(rare_types) == 0:
            return np.nan, {}, []

        per_type = {}

        for t in rare_types:

            idx = np.where(labels == t)[0]

            recovered = 0

            for i in idx:

                neigh = W[i].indices

                if len(neigh) == 0:
                    continue

                if np.any(labels[neigh] == t):
                    recovered += 1

            per_type[t] = recovered / len(idx)

        macro = np.mean(list(per_type.values()))

        return float(macro), per_type, rare_types


    import numpy as np

    # ============================================================
    # Moran's I for numeric vector
    # ============================================================

    def morans_I_numeric(x, W):
        """
        Moran's I for a numeric vector.
        """

        x = np.asarray(x, dtype=float)

        N = len(x)

        x_mean = np.mean(x)
        z = x - x_mean

        S0 = W.sum()

        denominator = np.sum(z ** 2)

        if S0 == 0 or denominator == 0:
            return np.nan

        numerator = z @ (W @ z)

        I = (N / S0) * (numerator / denominator)

        return float(I)


    # ============================================================
    # Moran's I for categorical labels
    # ============================================================

    def categorical_morans_I(
        labels,
        W,
        exclude_labels=("unlabeled",)
    ):
        """
        Calculate one-vs-rest Moran's I for every cell type.

        Example:
            Keratinocyte = 1, all others = 0
            Fibroblast   = 1, all others = 0
            etc.

        Returns
        -------
        macro_I:
            Mean Moran's I across cell types.

        results:
            Dictionary containing Moran's I for each cell type.
        """

        labels = np.asarray(labels).astype(str)

        classes = [
            x for x in np.unique(labels)
            if x not in exclude_labels
        ]

        results = {}

        for cell_type in classes:

            # binary indicator for this cell type
            binary = (labels == cell_type).astype(float)

            I = morans_I_numeric(
                binary,
                W
            )

            results[cell_type] = I

        # remove NaN values before averaging
        valid = [
            v for v in results.values()
            if np.isfinite(v)
        ]

        macro_I = (
            np.mean(valid)
            if len(valid) > 0
            else np.nan
        )

        return float(macro_I), results






In [ ]:
if RUN_COMPARISONS:
    import numpy as np
    import pandas as pd


    def evaluate_spatial_method(
        adata,
        label_col,
        W,
        method_name=None,
        rare_fraction=0.02,
        min_rare_cells=3
    ):
        """
        Evaluate one clustering/annotation method using four spatial metrics.

        Metrics
        -------
        Spatial coherence       : higher is better
        Spatial inconsistency   : lower is better
        Moran's I               : higher is better
        Rare-domain recovery    : higher is better
        """

        if label_col not in adata.obs.columns:
            raise ValueError(
                f"Column '{label_col}' not found in adata.obs"
            )

        labels = adata.obs[label_col].astype(str).values

        # ------------------------------------------
        # 1. Spatial coherence
        # ------------------------------------------
        coherence = spatial_coherence(
            labels,
            W
        )

        # ------------------------------------------
        # 2. Spatial inconsistency
        # ------------------------------------------
        inconsistency = spatial_inconsistency(
            labels,
            W
        )

        # ------------------------------------------
        # 3. Moran's I
        # ------------------------------------------
        moran_macro, moran_per_type = categorical_morans_I(
            labels,
            W
        )

        # ------------------------------------------
        # 4. Rare-domain recovery proxy
        # ------------------------------------------
        rare_recovery, rare_per_type, rare_types = (
            rare_domain_recovery_proxy(
                labels,
                W,
                rare_fraction=rare_fraction,
                min_cells=min_rare_cells
            )
        )

        if method_name is None:
            method_name = label_col

        result = {
            "Method": method_name,
            "Spatial coherence ↑": coherence,
            "Spatial inconsistency ↓": inconsistency,
            "Moran's I ↑": moran_macro,
            "Rare-domain recovery proxy ↑": rare_recovery
        }

        details = {
            "Moran_per_type": moran_per_type,
            "Rare_types": rare_types,
            "Rare_recovery_per_type": rare_per_type
        }

        return result, details

In [ ]:
if RUN_COMPARISONS:
    # ============================================================
    # SpaGCN — GSE208253 Oral Squamous Cell Carcinoma
    # Run in: conda activate spagcn_env
    # ============================================================

    import SpaGCN as spg
    import scanpy as sc
    import numpy as np
    import pandas as pd
    from scipy import sparse

    # ------------------------------------------------------------
    # 12 oral cancer samples
    # ------------------------------------------------------------
    samples = [
        "s1", "s2", "s3", "s4", "s5", "s6",
        "s7", "s8", "s9", "s10", "s11", "s12"
    ]

    # ------------------------------------------------------------
    # Storage for SpaGCN predictions
    # ------------------------------------------------------------
    spagcn_labels = pd.Series(
        index=adata.obs_names,
        dtype="object"
    )

    # Optional diagnostics
    spagcn_summary = []


    # ============================================================
    # Run SpaGCN sample-by-sample
    # ============================================================

    for s in samples:

        print("\n" + "=" * 60)
        print(f"Running SpaGCN: {s}")
        print("=" * 60)

        # --------------------------------------------------------
        # Extract one tissue section
        # --------------------------------------------------------
        ad = adata[
            adata.obs["sample"].astype(str) == s
        ].copy()

        print(
            f"{s}: {ad.n_obs} spots × {ad.n_vars} genes"
        )

        # --------------------------------------------------------
        # Preprocessing
        # --------------------------------------------------------
        sc.pp.filter_genes(
            ad,
            min_cells=3
        )

        # Prefer raw counts if a counts layer exists
        if "counts" in ad.layers:
            ad.X = ad.layers["counts"].copy()

        sc.pp.normalize_total(
            ad,
            target_sum=1e4
        )

        sc.pp.log1p(ad)

        # --------------------------------------------------------
        # Spatial coordinates
        # --------------------------------------------------------
        coords = np.asarray(
            ad.obsm["spatial"]
        )

        x = coords[:, 0]
        y = coords[:, 1]

        print(
            "Coordinate range:",
            f"x = {x.min():.1f}–{x.max():.1f},",
            f"y = {y.min():.1f}–{y.max():.1f}"
        )

        # --------------------------------------------------------
        # Spatial adjacency
        #
        # histology=False:
        # use spatial coordinates only
        # --------------------------------------------------------
        adj = spg.calculate_adj_matrix(
            x=x,
            y=y,
            histology=False
        )

        print(
            "Adjacency shape:",
            adj.shape
        )

        # --------------------------------------------------------
        # Search spatial parameter l
        # --------------------------------------------------------
        l = spg.search_l(
            p=0.5,
            adj=adj,
            start=0.01,
            end=1000,
            tol=0.01,
            max_run=100
        )

        print(
            f"{s}: selected l = {l}"
        )

        # --------------------------------------------------------
        # SpaGCN model
        # --------------------------------------------------------
        clf = spg.SpaGCN()

        clf.set_l(l)

        # Older SpaGCN internally expects dense expression data
        if sparse.issparse(ad.X):
            ad.X = ad.X.toarray().astype(
                np.float32
            )
        else:
            ad.X = np.asarray(
                ad.X,
                dtype=np.float32
            )

        # --------------------------------------------------------
        # Train
        # --------------------------------------------------------
        clf.train(
            ad,
            adj,
            init_spa=True,
            init="louvain",
            res=0.5,
            tol=5e-3,
            lr=0.05,
            max_epochs=200
        )

        # --------------------------------------------------------
        # Predict
        # --------------------------------------------------------
        pred, prob = clf.predict()

        pred = np.asarray(
            pred
        ).astype(str)

        # Store raw SpaGCN domain labels
        spagcn_labels.loc[
            ad.obs_names
        ] = pred

        # --------------------------------------------------------
        # Diagnostics
        # --------------------------------------------------------
        unique_clusters, cluster_counts = np.unique(
            pred,
            return_counts=True
        )

        print(
            f"\n{s}: SpaGCN domains = {len(unique_clusters)}"
        )

        print(
            pd.Series(
                cluster_counts,
                index=unique_clusters
            ).sort_index()
        )

        spagcn_summary.append({
            "sample": s,
            "n_spots": ad.n_obs,
            "n_genes": ad.n_vars,
            "n_clusters": len(unique_clusters),
            "l": l
        })


    # ============================================================
    # Add SpaGCN labels to merged oral-cancer AnnData
    # ============================================================

    adata.obs["label_SpaGCN"] = pd.Categorical(
        spagcn_labels.reindex(
            adata.obs_names
        )
    )


    # ============================================================
    # Check results
    # ============================================================

    print("\n" + "=" * 60)
    print("FINAL SpaGCN RESULTS")
    print("=" * 60)

    print(
        "\nTotal labeled spots:",
        adata.obs[
            "label_SpaGCN"
        ].notna().sum(),
        "/",
        adata.n_obs
    )

    print("\nOverall SpaGCN label distribution:")

    print(
        adata.obs[
            "label_SpaGCN"
        ].value_counts(
            dropna=False
        )
    )


    # ============================================================
    # Sample-wise domain distributions
    # ============================================================

    print("\nSample-wise SpaGCN clusters:")

    print(
        pd.crosstab(
            adata.obs["sample"],
            adata.obs["label_SpaGCN"]
        )
    )


    # ============================================================
    # Diagnostic summary
    # ============================================================

    spagcn_summary_df = pd.DataFrame(
        spagcn_summary
    )

    print("\nSpaGCN summary:")
    print(
        spagcn_summary_df.to_string(
            index=False
        )
    )


    # ============================================================
    # Save SpaGCN labels
    # ============================================================

    spagcn_output = adata.obs[
        [
            "sample",
            "barcode",
            "label_SpaGCN"
        ]
    ].copy()

    spagcn_output.to_csv(
        "GSE208253_SpaGCN_spatial_labels.csv",
        index=True
    )

    spagcn_summary_df.to_csv(
        "GSE208253_SpaGCN_summary.csv",
        index=False
    )

    print("\nSaved:")
    print("GSE208253_SpaGCN_spatial_labels.csv")
    print("GSE208253_SpaGCN_summary.csv")

In [ ]:
if RUN_COMPARISONS:
    # ============================================================
    # stLearn — GSE208253 Oral Squamous Cell Carcinoma
    # WITHOUT histology / morphology
    #
    # Run in:
    # conda activate stlearn_env
    # ============================================================

    import stlearn as st
    import scanpy as sc
    import pandas as pd
    import numpy as np


    # ------------------------------------------------------------
    # 12 OSCC samples
    # ------------------------------------------------------------
    samples = [
        "s1", "s2", "s3", "s4", "s5", "s6",
        "s7", "s8", "s9", "s10", "s11", "s12"
    ]


    # ------------------------------------------------------------
    # Output storage
    # ------------------------------------------------------------
    stlearn_labels = pd.Series(
        index=adata.obs_names.astype(str),
        dtype="object"
    )

    stlearn_summary = []


    # ============================================================
    # Run each tissue section independently
    # ============================================================

    for s in samples:

        print("\n" + "=" * 65)
        print(f"Running stLearn: {s}")
        print("=" * 65)

        # --------------------------------------------------------
        # 1. Get individual oral-cancer sample
        # --------------------------------------------------------
        ad_st = next(
            x.copy()
            for x in adatas
            if x.obs["sample"].iloc[0] == s
        )

        print(
            f"{s}: {ad_st.n_obs} spots × "
            f"{ad_st.n_vars} genes"
        )

        # --------------------------------------------------------
        # 2. Add coordinate columns expected by stLearn
        # --------------------------------------------------------
        coords = np.asarray(
            ad_st.obsm["spatial"]
        )

        ad_st.obs["imagecol"] = coords[:, 0]
        ad_st.obs["imagerow"] = coords[:, 1]

        print(
            "Spatial coordinate shape:",
            coords.shape
        )

        # --------------------------------------------------------
        # Preprocessing
        # --------------------------------------------------------
    
        st.pp.filter_genes(
            ad_st,
            min_cells=3
        )
    
        st.pp.normalize_total(
            ad_st,
            target_sum=1e4
        )
    
        st.pp.log1p(
            ad_st
        )
    
        # --------------------------------------------------------
        # PCA
        # --------------------------------------------------------
    
        n_pcs = min(
            30,
            ad_st.n_obs - 1,
            ad_st.n_vars - 1
        )
    
        st.em.run_pca(
            ad_st,
            n_comps=n_pcs
        )
    
        # --------------------------------------------------------
        # Compatibility workaround
        # --------------------------------------------------------
    
        ad_st.obsm["X_morphology"] = (
            ad_st.obsm["X_pca"].copy()
        )
    
        print(
            f"{s}: X_pca =",
            ad_st.obsm["X_pca"].shape
        )
    
        # --------------------------------------------------------
        # SME using physical distance + gene expression
        # --------------------------------------------------------
    
        ad_sme = ad_st.copy()
    
        st.spatial.sme.sme_normalize(
            ad_sme,
            use_data="raw",
            weights="weights_matrix_pd_gd"
        )
    
        print(
            f"{s}: SME normalized =",
            ad_sme.obsm["raw_SME_normalized"].shape
        )
    
        # --------------------------------------------------------
        # Use SME-normalized matrix
        # --------------------------------------------------------
    
        ad_sme.X = np.asarray(
            ad_sme.obsm["raw_SME_normalized"],
            dtype=np.float32
        )
    
        st.pp.scale(
            ad_sme
        )
    
        # --------------------------------------------------------
        # PCA after SME
        # --------------------------------------------------------
    
        n_pcs_sme = min(
            30,
            ad_sme.n_obs - 1,
            ad_sme.n_vars - 1
        )
    
        st.em.run_pca(
            ad_sme,
            n_comps=n_pcs_sme
        )
    
        # --------------------------------------------------------
        # Clustering
        # --------------------------------------------------------
    
        st.tl.clustering.kmeans(
            ad_sme,
            n_clusters=10,
            use_data="X_pca",
            key_added="stLearn_cluster"
        )
        # --------------------------------------------------------
        # 10. Check cluster distribution
        # --------------------------------------------------------
        print(
            f"\n{s} cluster distribution:"
        )

        print(
            ad_sme.obs[
                "stLearn_cluster"
            ]
            .value_counts()
            .sort_index()
        )

        # --------------------------------------------------------
        # 11. Extract labels
        # --------------------------------------------------------
        labels = (
            ad_sme.obs[
                "stLearn_cluster"
            ]
            .astype(str)
            .copy()
        )

        # --------------------------------------------------------
        # IMPORTANT:
        #
        # Your adatas already have names such as:
        #
        # s1_AAACAAGTATCTCCCA-1
        #
        # therefore DO NOT prefix s1 again.
        # --------------------------------------------------------

        common = labels.index.intersection(
            stlearn_labels.index
        )

        print(
            f"{s}: matched "
            f"{len(common)}/{len(labels)} spots"
        )

        stlearn_labels.loc[
            common
        ] = labels.loc[
            common
        ].values

        # --------------------------------------------------------
        # 12. Diagnostics
        # --------------------------------------------------------
        stlearn_summary.append({
            "sample": s,
            "n_spots": ad_sme.n_obs,
            "n_genes": ad_sme.n_vars,
            "n_clusters":
                ad_sme.obs[
                    "stLearn_cluster"
                ].nunique(),
            "n_matched":
                len(common),
            "match_percent":
                100 * len(common) / len(labels)
        })


    # ============================================================
    # Add final labels to merged oral object
    # ============================================================

    adata.obs[
        "label_stLearn"
    ] = pd.Categorical(
        stlearn_labels.reindex(
            adata.obs_names
        )
    )


    # ============================================================
    # Final checks
    # ============================================================

    print("\n" + "=" * 65)
    print("FINAL stLearn RESULTS")
    print("=" * 65)

    n_labeled = (
        adata.obs[
            "label_stLearn"
        ]
        .notna()
        .sum()
    )

    print(
        "\nLabeled spots:",
        n_labeled,
        "/",
        adata.n_obs
    )

    print(
        "\nOverall label distribution:"
    )

    print(
        adata.obs[
            "label_stLearn"
        ]
        .value_counts(
            dropna=False
        )
    )


    # ============================================================
    # Sample-wise distribution
    # ============================================================

    print(
        "\nSample-wise clusters:"
    )

    print(
        pd.crosstab(
            adata.obs[
                "sample"
            ],
            adata.obs[
                "label_stLearn"
            ]
        )
    )


    # ============================================================
    # Summary dataframe
    # ============================================================

    stlearn_summary_df = pd.DataFrame(
        stlearn_summary
    )

    print(
        "\nstLearn summary:"
    )

    print(
        stlearn_summary_df.to_string(
            index=False
        )
    )


    # ============================================================
    # Save labels
    # ============================================================

    stlearn_output = (
        adata.obs[
            [
                "sample",
                "barcode",
                "label_stLearn"
            ]
        ]
        .copy()
    )

    stlearn_output.to_csv(
        "GSE208253_stLearn_spatial_labels.csv",
        index=True
    )

    stlearn_summary_df.to_csv(
        "GSE208253_stLearn_summary.csv",
        index=False
    )

    print("\nSaved:")
    print(
        "GSE208253_stLearn_spatial_labels.csv"
    )
    print(
        "GSE208253_stLearn_summary.csv"
    )

In [ ]:
if RUN_COMPARISONS:
    # ============================================================
    # FINAL SPATIAL BENCHMARK
    # scAICME vs Leiden vs SpaGCN vs stLearn
    # ============================================================

    import numpy as np
    import pandas as pd

    from scipy import sparse
    from sklearn.neighbors import NearestNeighbors


    # ============================================================
    # 1. LOAD SAVED LABEL FILES
    # ============================================================

    scaicme_df = pd.read_csv("scAICME_spatial_oral_labels.csv")
    leiden_df  = pd.read_csv("Leiden_spatial_labels.csv")
    spagcn_df  = pd.read_csv("GSE208253_SpaGCN_spatial_labels.csv")
    stlearn_df = pd.read_csv("GSE208253_stLearn_spatial_labels.csv")


    print("scAICME:", scaicme_df.shape)
    print("Leiden :", leiden_df.shape)
    print("SpaGCN :", spagcn_df.shape)
    print("stLearn:", stlearn_df.shape)

In [ ]:
if RUN_COMPARISONS:
    # --------------------------------------------------
    # Create spot_key in all four dataframes
    # --------------------------------------------------

    for df in [
        scaicme_df,
        leiden_df,
        spagcn_df,
        stlearn_df
    ]:
        df["sample"] = df["sample"].astype(str)
        df["barcode"] = df["barcode"].astype(str)

        df["spot_key"] = (
            df["sample"]
            + "::"
            + df["barcode"]
        )

In [ ]:
if RUN_COMPARISONS:
    comparison_labels = (
        scaicme_df[
            [
                "sample",
                "barcode",
                "spot_key",
                "label_consensus"
            ]
        ]
        .merge(
            leiden_df[
                [
                    "spot_key",
                    "label_leiden"
                ]
            ],
            on="spot_key",
            how="inner"
        )
        .merge(
            spagcn_df[
                [
                    "spot_key",
                    "label_SpaGCN"
                ]
            ],
            on="spot_key",
            how="inner"
        )
        .merge(
            stlearn_df[
                [
                    "spot_key",
                    "label_stLearn"
                ]
            ],
            on="spot_key",
            how="inner"
        )
    )

In [ ]:
if RUN_COMPARISONS:
    print(comparison_labels.shape)

    print(
        comparison_labels[
            [
                "label_consensus",
                "label_leiden",
                "label_SpaGCN",
                "label_stLearn"
            ]
        ].notna().sum()
    )

In [ ]:
if RUN_COMPARISONS:
    adata.obs["spot_key"] = (
        adata.obs["sample"].astype(str)
        + "::"
        + adata.obs["barcode"].astype(str)
    )

    keep = (
        adata.obs["spot_key"]
        .isin(comparison_labels["spot_key"])
    )

    adata_compare = adata[
        keep
    ].copy()

    print(adata_compare)

In [ ]:
if RUN_COMPARISONS:
    lab = comparison_labels.set_index(
        "spot_key"
    )

In [ ]:
if RUN_COMPARISONS:
    for col in [
        "label_consensus",
        "label_leiden",
        "label_SpaGCN",
        "label_stLearn"
    ]:

        adata_compare.obs[col] = (
            adata_compare.obs["spot_key"]
            .map(lab[col])
            .astype("category")
        )

In [ ]:
if RUN_COMPARISONS:
    for col in [
        "label_consensus",
        "label_leiden",
        "label_SpaGCN",
        "label_stLearn"
    ]:

        print("\n", col)

        print(
            adata_compare.obs[col]
            .value_counts(dropna=False)
        )

In [ ]:
if RUN_COMPARISONS:
    print(
        adata_compare.obs[
            [
                "label_consensus",
                "label_leiden",
                "label_SpaGCN",
                "label_stLearn"
            ]
        ].isna().sum()
    )

In [ ]:
if RUN_COMPARISONS:
    W_compare = build_samplewise_spatial_graph(
        adata_compare,
        sample_col="sample",
        spatial_key="spatial",
        n_neighbors=6
    )

    print(W_compare.shape)

In [ ]:
if RUN_COMPARISONS:
    methods = {
        "scAICME": "label_consensus",
        "Leiden": "label_leiden",
        "SpaGCN": "label_SpaGCN",
        "stLearn": "label_stLearn"
    }

    comparison_results = []
    comparison_details = {}

    for method_name, label_col in methods.items():

        result, details = evaluate_spatial_method(
            adata=adata_compare,
            label_col=label_col,
            W=W_compare,
            method_name=method_name,
            rare_fraction=0.02,
            min_rare_cells=3
        )

        comparison_results.append(result)
        comparison_details[method_name] = details


    final_results = pd.DataFrame(
        comparison_results
    )

    final_results = final_results[
        [
            "Method",
            "Spatial coherence ↑",
            "Spatial inconsistency ↓",
            "Moran's I ↑",
        ]
    ]

    final_results

In [ ]:
if RUN_COMPARISONS:
    final_results.to_csv(
        "Spatial_methods_final_Oral_results.csv",
        index=False
    )